# Football Data Downloader

Downloads historical match data from football-data.co.uk into an organized
folder structure.

**Resumable by design**: each cell checks which files already exist on disk
before downloading, and skips anything already there. If your connection
drops mid-run, just re-run the same cell — it picks up where it left off
instead of starting over.

Run cells top to bottom. Start with the small test in Cell 4 before
uncommenting the bigger pulls further down.

In [ ]:
import os
import time
import requests

BASE_URL = "https://www.football-data.co.uk"
# When running LOCALLY, use: OUTPUT_DIR = "data/raw"
# When running INSIDE DATABRICKS, point at your Volume instead:
OUTPUT_DIR = "/Volumes/workspace/default/football_raw"


## League & season definitions

In [ ]:
# Division names match scripts/rename_league_folders.py exactly, so a fresh
# download in Databricks produces the SAME folder names as the local copy.
MAIN_LEAGUES = {
    "england": {"name": "England", "divisions": {
        "E0": "Premier League", "E1": "Championship",
        "E2": "League One", "E3": "League Two", "EC": "National League"}},
    "scotland": {"name": "Scotland", "divisions": {
        "SC0": "Scottish Premiership", "SC1": "Scottish Championship",
        "SC2": "Scottish League One", "SC3": "Scottish League Two"}},
    "germany": {"name": "Germany", "divisions": {
        "D1": "Bundesliga", "D2": "2. Bundesliga"}},
    "italy": {"name": "Italy", "divisions": {
        "I1": "Serie A", "I2": "Serie B"}},
    "spain": {"name": "Spain", "divisions": {
        "SP1": "La Liga", "SP2": "La Liga 2"}},
    "france": {"name": "France", "divisions": {
        "F1": "Ligue 1", "F2": "Ligue 2"}},
    "netherlands": {"name": "Netherlands", "divisions": {"N1": "Eredivisie"}},
    "belgium": {"name": "Belgium", "divisions": {"B1": "Belgian Pro League"}},
    "portugal": {"name": "Portugal", "divisions": {"P1": "Primeira Liga"}},
    "turkey": {"name": "Turkey", "divisions": {"T1": "Super Lig"}},
    "greece": {"name": "Greece", "divisions": {"G1": "Super League Greece"}},
}

# (url_code, display_label) — "/" can't be used in filenames, so folders
# use "2025-26" rather than "2025/26"
MAIN_SEASONS = [
    ("2526", "2025-26"), ("2425", "2024-25"), ("2324", "2023-24"),
    ("2223", "2022-23"), ("2122", "2021-22"), ("2021", "2020-21"),
    ("1920", "2019-20"), ("1819", "2018-19"), ("1718", "2017-18"),
    ("1617", "2016-17"), ("1516", "2015-16"), ("1415", "2014-15"),
    ("1314", "2013-14"), ("1213", "2012-13"), ("1112", "2011-12"),
    ("1011", "2010-11"), ("0910", "2009-10"), ("0809", "2008-09"),
    ("0708", "2007-08"), ("0607", "2006-07"), ("0506", "2005-06"),
]

EXTRA_LEAGUES = {
    "argentina": ("ARG", "Primera Division"), "austria": ("AUT", "Bundesliga"),
    "brazil": ("BRA", "Serie A"), "china": ("CHN", "Super League"),
    "denmark": ("DNK", "Superliga"), "finland": ("FIN", "Veikkausliiga"),
    "ireland": ("IRL", "Premier Division"), "japan": ("JPN", "J-League"),
    "mexico": ("MEX", "Liga MX"), "norway": ("NOR", "Eliteserien"),
    "poland": ("POL", "Ekstraklasa"), "romania": ("ROU", "Liga 1"),
    "russia": ("RUS", "Premier League"), "sweden": ("SWE", "Allsvenskan"),
    "switzerland": ("SWZ", "Super League"), "usa": ("USA", "MLS"),
}


## Resumable download function

`skip_existing=True` (the default) means: if the destination file already
exists on disk and is non-empty, it's assumed already downloaded and gets
skipped. This is what makes re-running safe after a dropped connection.

In [ ]:
def download_file(url: str, dest_path: str, skip_existing: bool = True) -> str:
    """Returns one of: "skipped", "ok", "miss", "error" """
    if skip_existing and os.path.exists(dest_path) and os.path.getsize(dest_path) > 0:
        return "skipped"
    try:
        resp = requests.get(url, timeout=20)
        if resp.status_code == 200 and len(resp.content) > 0:
            os.makedirs(os.path.dirname(dest_path), exist_ok=True)
            with open(dest_path, "wb") as f:
                f.write(resp.content)
            return "ok"
        return "miss"
    except requests.RequestException:
        return "error"


In [ ]:
def download_main_leagues(seasons=None, countries=None, skip_existing=True):
    seasons = seasons or MAIN_SEASONS
    countries = countries or list(MAIN_LEAGUES.keys())
    counts = {"ok": 0, "skipped": 0, "miss": 0, "error": 0}

    for country_key in countries:
        country = MAIN_LEAGUES[country_key]
        print(f"\n{country['name']}")
        for div_code, div_name in country["divisions"].items():
            for url_code, label in seasons:
                url = f"{BASE_URL}/mmz4281/{url_code}/{div_code}.csv"
                dest = os.path.join(OUTPUT_DIR, "main_leagues", country_key, div_name, f"{label}.csv")
                result = download_file(url, dest, skip_existing)
                counts[result] += 1
                tag = {"ok": "OK   ", "skipped": "SKIP ", "miss": "MISS ", "error": "ERROR"}[result]
                print(f"  {tag} {div_name} ({div_code}) — {label}")
                if result == "ok":
                    time.sleep(0.3)

    print(f"\nDone: {counts['ok']} downloaded, {counts['skipped']} already had, "
          f"{counts['miss']} missing, {counts['error']} errors")
    return counts


In [ ]:
def download_extra_leagues(countries=None, skip_existing=True):
    countries = countries or list(EXTRA_LEAGUES.keys())
    counts = {"ok": 0, "skipped": 0, "miss": 0, "error": 0}

    print("Extra Leagues")
    for country_key in countries:
        code_, league_name = EXTRA_LEAGUES[country_key]
        url = f"{BASE_URL}/new/{code_}.csv"
        dest = os.path.join(OUTPUT_DIR, "extra_leagues", f"{code_}.csv")
        result = download_file(url, dest, skip_existing)
        counts[result] += 1
        tag = {"ok": "OK   ", "skipped": "SKIP ", "miss": "MISS ", "error": "ERROR"}[result]
        print(f"  {tag} {country_key.title()} — {league_name} ({code_})")
        if result == "ok":
            time.sleep(0.3)

    print(f"\nDone: {counts['ok']} downloaded, {counts['skipped']} already had, "
          f"{counts['miss']} missing, {counts['error']} errors")
    return counts


## Step 1: Small test — Premier League, last 2 seasons

Run this first. Confirms everything works before scaling up.

In [ ]:
download_main_leagues(
    seasons=[("2526", "2025-26"), ("2425", "2024-25")],
    countries=["england"],
)

## Step 2: Expand — once Step 1 works

Uncomment to pull more. **Safe to re-run this cell as many times as you
need** — already-downloaded files get skipped instantly, only missing ones
get fetched. If your network cuts out partway through, just run it again.

In [ ]:
# Full England history, all divisions:
# download_main_leagues(countries=["england"])

# All main leagues, current + last season only (lighter):
# download_main_leagues(seasons=[("2526", "2025-26"), ("2425", "2024-25")])

# Everything (this is hundreds of requests — only do this once you're
# confident the small tests above worked cleanly):
# download_main_leagues()
# download_extra_leagues()

## Step 3: Verify what you actually have on disk

In [ ]:
for root, dirs, files in os.walk(OUTPUT_DIR):
    csvs = [f for f in files if f.endswith(".csv")]
    if csvs:
        print(f"{root}: {len(csvs)} file(s)")